# Playground Series S6E4 — Predicting Irrigation Need
## EDA + CatBoost Pipeline

**Task**: Multi-class classification: `Low` / `Medium` / `High` irrigation need  
**Metric**: Balanced Accuracy (class-imbalance aware)  
**Strategy**: Deep EDA → Feature Engineering → CatBoost with native categoricals

---
### Research Agenda
1. Data quality & basic statistics
2. Target class distribution (imbalance analysis)
3. Numerical feature distributions by class
4. Categorical feature class proportions
5. Correlation & multicollinearity
6. Outlier detection
7. Feature interactions & domain insights
8. Feature engineering guided by EDA
9. CatBoost with 5-fold stratified CV
10. Optuna hyperparameter search
11. Submission

## 0. Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['optuna', 'catboost']:
    try:
        __import__(pkg)
    except ImportError:
        _install(pkg)

print('Environment ready')

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report

from catboost import CatBoostClassifier, Pool as CatPool
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# GPU detection via PyTorch
try:
    import torch
    USE_GPU = torch.cuda.is_available()
    if USE_GPU:
        print('GPU :', torch.cuda.get_device_name(0))
        print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')
except ImportError:
    USE_GPU = False

RANDOM_STATE = 42
N_FOLDS = 5
TARGET = 'Irrigation_Need'
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
PALETTE = {'Low': '#4C9BE8', 'Medium': '#F5A623', 'High': '#E84C4C'}

CAT_COLS = [
    'Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
    'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region'
]
NUM_COLS = [
    'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
    'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
    'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm'
]

print('USE_GPU =', USE_GPU)
print('Numerical:', len(NUM_COLS), '| Categorical:', len(CAT_COLS))

## 1. Load Data

Auto-detect Kaggle competition path vs local `datasets/` folder.

In [ ]:
_candidates = [
    '/kaggle/input/playground-series-s6e4',
    '/kaggle/input/competitions/playground-series-s6e4',
]
KAGGLE_INPUT = next((p for p in _candidates if os.path.exists(p)), None)
DATA_DIR = KAGGLE_INPUT if KAGGLE_INPUT else 'datasets'

print('Data directory:', DATA_DIR)

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
sub   = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print('Train :', train.shape)
print('Test  :', test.shape)
print()
print('Class distribution:')
print(train[TARGET].value_counts())
print()
print('Class ratios:')
print(train[TARGET].value_counts(normalize=True).round(4))

## 2. Data Quality

Check for missing values, data types, and basic statistics before any modelling.

In [ ]:
print('=' * 50)
print('MISSING VALUES')
print('=' * 50)
miss_train = train.isnull().sum()
miss_test  = test.isnull().sum()
quality_df = pd.DataFrame({
    'dtype'        : train.dtypes,
    'missing_train': miss_train,
    'missing_test' : miss_test,
    'unique_train' : train.nunique(),
})
print(quality_df)
print()
print('Total missing — train:', miss_train.sum(), '| test:', miss_test.sum())

In [ ]:
print('Descriptive statistics — numerical features:')
train[NUM_COLS].describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std', '50%'])

## 3. Class Imbalance Analysis

`High` irrigation need accounts for only ~3.3% of training samples — the core challenge.  
All models must account for this via class weighting.

In [ ]:
counts = train[TARGET].value_counts().reindex(['Low', 'Medium', 'High'])
pcts   = counts / counts.sum() * 100

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Bar chart
bars = axes[0].bar(counts.index, counts.values,
                   color=[PALETTE[c] for c in counts.index], edgecolor='white', linewidth=1.2)
axes[0].set_title('Sample Counts', fontweight='bold')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{val:,}', ha='center', fontweight='bold')

# Pie chart
pie_labels = [c + '\n' + f'{p:.1f}%' for c, p in zip(pcts.index, pcts.values)]
axes[1].pie(pcts.values, labels=pie_labels,
            colors=[PALETTE[c] for c in pcts.index], startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Share', fontweight='bold')

# Imbalance ratios
ir_medium = counts['Low'] / counts['Medium']
ir_high   = counts['Low'] / counts['High']
axes[2].barh(['Low:High ratio', 'Low:Medium ratio'], [ir_high, ir_medium],
             color=['#E84C4C', '#F5A623'])
axes[2].set_title('Imbalance Ratios', fontweight='bold')
for i, v in enumerate([ir_high, ir_medium]):
    axes[2].text(v + 0.3, i, f'{v:.1f}x', va='center', fontweight='bold')

plt.suptitle('Target: Irrigation Need — Class Imbalance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Imbalance ratio Low:High   =', round(ir_high, 1), 'x')
print('Imbalance ratio Low:Medium =', round(ir_medium, 1), 'x')
print()
print("=> auto_class_weights='Balanced' must be used in CatBoost")

## 4. Numerical Feature Distributions by Class

Overlapping density plots reveal which features separate classes best.  
Strong separation = high predictive power for that feature.

In [ ]:
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(3, 4, figsize=(20, 13))
axes = axes.flatten()

for i, col in enumerate(NUM_COLS):
    for cls in ['Low', 'Medium', 'High']:
        data = train.loc[train[TARGET] == cls, col].dropna()
        axes[i].hist(data, bins=45, alpha=0.5, density=True, label=cls, color=PALETTE[cls])
        if len(data) > 10:
            kde = gaussian_kde(data, bw_method=0.3)
            x_range = np.linspace(data.min(), data.max(), 200)
            axes[i].plot(x_range, kde(x_range), color=PALETTE[cls], linewidth=1.8)
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=7)

for j in range(len(NUM_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions by Class (density + KDE)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Kruskal-Wallis H-test: do classes have significantly different distributions?
print('Kruskal-Wallis H-test (p < 0.05 => significant class differences):')
print('-' * 60)
kw_results = []
for col in NUM_COLS:
    groups = [train.loc[train[TARGET] == cls, col].dropna() for cls in ['Low', 'Medium', 'High']]
    h_stat, p_val = stats.kruskal(*groups)
    kw_results.append({'Feature': col, 'H-stat': round(h_stat, 2), 'p-value': round(p_val, 6),
                       'Significant': 'YES' if p_val < 0.05 else 'no'})

kw_df = pd.DataFrame(kw_results).sort_values('H-stat', ascending=False)
print(kw_df.to_string(index=False))

## 5. Categorical Feature Analysis

Stacked bar charts show how class proportions shift across each category.  
Unequal proportions across categories = useful discriminative signal.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

for i, col in enumerate(CAT_COLS):
    ct = pd.crosstab(train[col], train[TARGET], normalize='index')
    ct = ct[[c for c in ['Low', 'Medium', 'High'] if c in ct.columns]]
    ct.plot(kind='bar', stacked=True, ax=axes[i],
            color=[PALETTE[c] for c in ct.columns],
            edgecolor='white', linewidth=0.5)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=40, labelsize=7)
    axes[i].legend(fontsize=7, loc='lower right')
    axes[i].set_ylabel('Proportion')
    axes[i].set_ylim(0, 1.05)

plt.suptitle('Categorical Features: Class Proportion by Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2, _, _, _ = chi2_contingency(ct)
    n = ct.values.sum()
    r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

print("Cramer's V — association with target (higher = stronger):")
print('-' * 50)
cv_results = [(col, cramers_v(train[col], train[TARGET])) for col in CAT_COLS]
cv_df = pd.DataFrame(cv_results, columns=['Feature', "Cramer's V"])
cv_df = cv_df.sort_values("Cramer's V", ascending=False)
print(cv_df.to_string(index=False))

## 6. Correlation Analysis

Pearson correlation between numerical features.  
High correlation (>0.7) between features may cause redundancy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Correlation heatmap
corr = train[NUM_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.4, ax=axes[0],
            annot_kws={'size': 8})
axes[0].set_title('Pearson Correlation — Numerical Features', fontweight='bold', fontsize=12)

# Correlation with target
target_enc = train[TARGET].map({'Low': 0, 'Medium': 1, 'High': 2})
pb_corrs = {}
for col in NUM_COLS:
    r, p = stats.pearsonr(train[col].fillna(train[col].median()), target_enc)
    pb_corrs[col] = r

pb_series = pd.Series(pb_corrs).sort_values()
colors = ['#E84C4C' if v < 0 else '#4C9BE8' for v in pb_series.values]
axes[1].barh(pb_series.index, pb_series.values, color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Pearson Correlation with Target', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Correlation')

plt.tight_layout()
plt.show()

print('Highly correlated pairs (|r| > 0.5):')
high_corr = [(c1, c2, corr.loc[c1, c2])
             for c1 in NUM_COLS for c2 in NUM_COLS
             if c1 < c2 and abs(corr.loc[c1, c2]) > 0.5]
for c1, c2, r in sorted(high_corr, key=lambda x: -abs(x[2])):
    print(f'  {c1} <-> {c2}: {r:.3f}')

## 7. Outlier Analysis

Box plots + IQR-based outlier counts per class.  
Outliers in `High` class (only ~3.3% of data) can be particularly influential.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 13))
axes = axes.flatten()
outlier_summary = []

for i, col in enumerate(NUM_COLS):
    class_data = [train.loc[train[TARGET] == cls, col].dropna() for cls in ['Low', 'Medium', 'High']]
    bp = axes[i].boxplot(class_data, labels=['Low', 'Med', 'High'],
                         patch_artist=True, notch=True)
    for patch, cls in zip(bp['boxes'], ['Low', 'Medium', 'High']):
        patch.set_facecolor(PALETTE[cls])
        patch.set_alpha(0.7)
    axes[i].set_title(col, fontweight='bold')
    q1, q3 = train[col].quantile(0.25), train[col].quantile(0.75)
    iqr = q3 - q1
    n_out = ((train[col] < q1 - 1.5*iqr) | (train[col] > q3 + 1.5*iqr)).sum()
    outlier_summary.append({'Feature': col, 'IQR_outliers': n_out,
                            'Outlier_%': round(n_out / len(train) * 100, 2)})

for j in range(len(NUM_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Outlier Analysis — Box Plots by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

out_df = pd.DataFrame(outlier_summary).sort_values('IQR_outliers', ascending=False)
print('IQR-based outliers:')
print(out_df.to_string(index=False))

## 8. Feature Engineering

Domain-inspired agronomic features derived from EDA insights:

| Feature | Formula | Agronomic Intuition |
|---------|---------|---------------------|
| `Water_Stress` | Temp / (Soil_Moisture + ε) | High temp + low moisture → irrigation needed |
| `Evapotranspiration_Proxy` | Sunlight × Temp × (1 − Humidity) | Atmospheric water demand |
| `Moisture_Deficit` | 65 − Soil_Moisture | Distance from ideal field capacity |
| `Total_Water_Input` | Rainfall + Previous_Irrigation | Total incoming water |
| `Wind_Evap` | Wind × Temp / (Humidity + 1) | Wind-driven evaporation |
| `EC_pH_interaction` | EC × pH | Soil chemistry interaction |
| `Carbon_Moisture` | OC × Soil_Moisture | Soil organic matter water retention |
| `Soil_Quality` | OC / (EC + ε) | Composite soil health index |

In [ ]:
def engineer_features(df):
    df = df.copy()
    # Water availability
    df['Water_Stress']              = df['Temperature_C'] / (df['Soil_Moisture'] + 1e-3)
    df['Rain_vs_Prev']              = df['Rainfall_mm'] / (df['Previous_Irrigation_mm'] + 1e-3)
    df['Moisture_Deficit']          = 65 - df['Soil_Moisture']
    df['Total_Water_Input']         = df['Rainfall_mm'] + df['Previous_Irrigation_mm']
    df['Prev_Irrig_Moisture_ratio'] = df['Previous_Irrigation_mm'] / (df['Soil_Moisture'] + 1e-3)
    # Atmospheric demand
    df['Temp_Humidity']            = df['Temperature_C'] * (1 - df['Humidity'] / 100)
    df['Evapotranspiration_Proxy'] = df['Sunlight_Hours'] * df['Temp_Humidity']
    df['Wind_Evap']                = df['Wind_Speed_kmh'] * df['Temperature_C'] / (df['Humidity'] + 1)
    df['Rain_per_hour']            = df['Rainfall_mm'] / (df['Sunlight_Hours'] + 1)
    # Soil quality
    df['EC_pH_interaction'] = df['Electrical_Conductivity'] * df['Soil_pH']
    df['Carbon_Moisture']   = df['Organic_Carbon'] * df['Soil_Moisture']
    df['Soil_Quality']      = df['Organic_Carbon'] / (df['Electrical_Conductivity'] + 1e-3)
    return df

train = engineer_features(train)
test  = engineer_features(test)

NEW_FEATURES = [
    'Water_Stress', 'Rain_vs_Prev', 'Temp_Humidity', 'Evapotranspiration_Proxy',
    'Moisture_Deficit', 'EC_pH_interaction', 'Carbon_Moisture', 'Wind_Evap',
    'Rain_per_hour', 'Prev_Irrig_Moisture_ratio', 'Total_Water_Input', 'Soil_Quality'
]
FEATURE_COLS = NUM_COLS + NEW_FEATURES + CAT_COLS
print('Features:', len(FEATURE_COLS), 'total')
print(' ', len(NUM_COLS), 'raw numerical')
print(' ', len(NEW_FEATURES), 'engineered')
print(' ', len(CAT_COLS), 'categorical')

In [ ]:
# Visualise engineered features distributions by class
fig, axes = plt.subplots(3, 4, figsize=(20, 13))
axes = axes.flatten()

for i, col in enumerate(NEW_FEATURES):
    for cls in ['Low', 'Medium', 'High']:
        data = train.loc[train[TARGET] == cls, col].dropna()
        axes[i].hist(data, bins=40, alpha=0.5, density=True, label=cls, color=PALETTE[cls])
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=7)

plt.suptitle('Engineered Feature Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Prepare for CatBoost

CatBoost handles string categoricals natively via ordered target statistics —  
no label encoding needed, avoiding target leakage.

In [ ]:
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
target_inv = {v: k for k, v in target_map.items()}

y      = train[TARGET].map(target_map).values
X      = train[FEATURE_COLS].copy()
X_test = test[FEATURE_COLS].copy()

# CatBoost requires categoricals as strings
for col in CAT_COLS:
    X[col]      = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print('X      :', X.shape)
print('y      :', y.shape)
print('Classes:', dict(zip(target_map.keys(), np.bincount(y))))
print('Cat cols:', CAT_COLS)

## 10. CatBoost — Baseline 5-Fold CV

**Why CatBoost?**
- Native categorical support → no encoding leakage
- `auto_class_weights='Balanced'` handles the ~30x imbalance automatically
- Ordered boosting reduces overfitting on small minority class folds
- GPU acceleration (~10x speedup on Kaggle T4)

In [ ]:
cat_params_base = {
    'iterations'           : 1500,
    'learning_rate'        : 0.04,
    'depth'                : 7,
    'l2_leaf_reg'          : 3.0,
    'bagging_temperature'  : 0.5,
    'random_strength'      : 1.0,
    'auto_class_weights'   : 'Balanced',
    'loss_function'        : 'MultiClass',
    'eval_metric'          : 'Accuracy',
    'early_stopping_rounds': 100,
    'random_seed'          : RANDOM_STATE,
    'verbose'              : 0,
    'task_type'            : 'GPU' if USE_GPU else 'CPU',
}

oof_cat  = np.zeros((len(X), 3))
test_cat = np.zeros((len(X_test), 3))
cat_scores = []
cat_models = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr_df = X.iloc[tr_idx]
    X_va_df = X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    train_pool = CatPool(X_tr_df, label=y_tr, cat_features=CAT_COLS)
    val_pool   = CatPool(X_va_df, label=y_va, cat_features=CAT_COLS)
    test_pool  = CatPool(X_test,              cat_features=CAT_COLS)

    m = CatBoostClassifier(**cat_params_base)
    m.fit(train_pool, eval_set=val_pool, use_best_model=True)

    oof_cat[va_idx] = m.predict_proba(val_pool)
    test_cat        += m.predict_proba(test_pool) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_cat[va_idx].argmax(1))
    cat_scores.append(score)
    cat_models.append(m)
    print(f'  Fold {fold+1}: BalAcc={score:.5f}  (best_iter={m.best_iteration_})')

cat_cv = balanced_accuracy_score(y, oof_cat.argmax(1))
print()
print('CatBoost Baseline OOF:', round(cat_cv, 5))
print('Fold std            :', round(float(np.std(cat_scores)), 5))

## 11. Model Diagnostics

Confusion matrix and per-class metrics on OOF predictions.

In [ ]:
oof_preds  = oof_cat.argmax(1)
class_names = ['Low', 'Medium', 'High']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Normalised confusion matrix
cm      = confusion_matrix(y, oof_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0],
            linewidths=0.5)
axes[0].set_title('OOF Confusion Matrix (normalised)', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

# Per-class recall
recall_per_class = cm_norm.diagonal()
axes[1].bar(class_names, recall_per_class, color=[PALETTE[c] for c in class_names])
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Per-class Recall', fontweight='bold')
axes[1].set_ylabel('Recall')
for i, v in enumerate(recall_per_class):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
axes[1].axhline(y=cat_cv, color='red', linestyle='--', label=f'BalAcc={cat_cv:.4f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Classification Report (OOF):')
print(classification_report(y, oof_preds, target_names=class_names))

## 12. Feature Importance

Average importances across all 5 folds.  
Colours: blue = raw numerical, red = engineered, orange = categorical.

In [ ]:
from matplotlib.patches import Patch

importances = np.zeros(len(FEATURE_COLS))
for m in cat_models:
    importances += m.get_feature_importance() / len(cat_models)

imp_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': importances})
imp_df = imp_df.sort_values('Importance', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Top 20 features
top20 = imp_df.head(20)
colors = ['#E84C4C' if f in NEW_FEATURES else '#4C9BE8' if f in NUM_COLS else '#F5A623'
          for f in top20['Feature']]
axes[0].barh(top20['Feature'][::-1], top20['Importance'][::-1], color=colors[::-1])
axes[0].set_title('Top 20 Feature Importances', fontweight='bold')
axes[0].set_xlabel('Importance')
legend_elements = [Patch(facecolor='#4C9BE8', label='Raw Numerical'),
                   Patch(facecolor='#E84C4C', label='Engineered'),
                   Patch(facecolor='#F5A623', label='Categorical')]
axes[0].legend(handles=legend_elements, loc='lower right')

# Importance share by feature type
eng_imp = imp_df[imp_df['Feature'].isin(NEW_FEATURES)]['Importance'].sum()
raw_imp = imp_df[imp_df['Feature'].isin(NUM_COLS)]['Importance'].sum()
cat_imp = imp_df[imp_df['Feature'].isin(CAT_COLS)]['Importance'].sum()
total   = eng_imp + raw_imp + cat_imp
pie_labels = [
    'Engineered\n' + f'{eng_imp/total:.1%}',
    'Raw Numerical\n' + f'{raw_imp/total:.1%}',
    'Categorical\n' + f'{cat_imp/total:.1%}',
]
axes[1].pie([eng_imp, raw_imp, cat_imp], labels=pie_labels,
            colors=['#E84C4C', '#4C9BE8', '#F5A623'], startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Importance Share by Feature Type', fontweight='bold')

plt.tight_layout()
plt.show()
print(imp_df.head(20).to_string(index=False))

## 13. Optuna Hyperparameter Tuning

TPE sampler, 40 trials with 3-fold inner CV for speed.  
Optimises Balanced Accuracy on OOF predictions — no leakage.

In [ ]:
skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def cat_objective(trial):
    params = {
        'iterations'           : trial.suggest_int('iterations', 800, 2500),
        'learning_rate'        : trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'depth'                : trial.suggest_int('depth', 5, 10),
        'l2_leaf_reg'          : trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bagging_temperature'  : trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength'      : trial.suggest_float('random_strength', 0.5, 3.0),
        'auto_class_weights'   : 'Balanced',
        'loss_function'        : 'MultiClass',
        'eval_metric'          : 'Accuracy',
        'early_stopping_rounds': 80,
        'random_seed'          : RANDOM_STATE,
        'verbose'              : 0,
        'task_type'            : 'GPU' if USE_GPU else 'CPU',
    }
    oof = np.zeros((len(X), 3))
    for tr_idx, va_idx in skf3.split(X, y):
        tr_pool = CatPool(X.iloc[tr_idx], label=y[tr_idx], cat_features=CAT_COLS)
        va_pool = CatPool(X.iloc[va_idx], label=y[va_idx], cat_features=CAT_COLS)
        m = CatBoostClassifier(**params)
        m.fit(tr_pool, eval_set=va_pool, use_best_model=True)
        oof[va_idx] = m.predict_proba(va_pool)
    return balanced_accuracy_score(y, oof.argmax(1))

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study.optimize(cat_objective, n_trials=40, show_progress_bar=True)

print()
print('Best OOF score:', round(study.best_value, 5))
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

## 14. Retrain Tuned CatBoost (5-Fold)

Use optimised hyperparameters for the final submission model.

In [ ]:
best_params = study.best_params.copy()
best_params.update({
    'auto_class_weights'   : 'Balanced',
    'loss_function'        : 'MultiClass',
    'eval_metric'          : 'Accuracy',
    'early_stopping_rounds': 100,
    'random_seed'          : RANDOM_STATE,
    'verbose'              : 0,
    'task_type'            : 'GPU' if USE_GPU else 'CPU',
})

oof_tuned  = np.zeros((len(X), 3))
test_tuned = np.zeros((len(X_test), 3))
tuned_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr_df = X.iloc[tr_idx]
    X_va_df = X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    train_pool = CatPool(X_tr_df, label=y_tr, cat_features=CAT_COLS)
    val_pool   = CatPool(X_va_df, label=y_va, cat_features=CAT_COLS)
    test_pool  = CatPool(X_test,              cat_features=CAT_COLS)

    m = CatBoostClassifier(**best_params)
    m.fit(train_pool, eval_set=val_pool, use_best_model=True)

    oof_tuned[va_idx] = m.predict_proba(val_pool)
    test_tuned        += m.predict_proba(test_pool) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_tuned[va_idx].argmax(1))
    tuned_scores.append(score)
    print(f'  Fold {fold+1}: BalAcc={score:.5f}  (best_iter={m.best_iteration_})')

tuned_cv = balanced_accuracy_score(y, oof_tuned.argmax(1))
print()
print('Tuned CatBoost OOF:', round(tuned_cv, 5))
print('Baseline OOF      :', round(cat_cv, 5))
print('Delta             :', round(tuned_cv - cat_cv, 5))

## 15. Final Comparison & Submission

Choose the better model based on OOF Balanced Accuracy.

In [ ]:
final_df = pd.DataFrame({
    'Model': ['CatBoost Baseline', 'CatBoost Tuned'],
    'OOF_BalAcc': [cat_cv, tuned_cv],
}).sort_values('OOF_BalAcc', ascending=False).reset_index(drop=True)

print('=' * 40)
print('FINAL OOF SCORES')
print('=' * 40)
print(final_df.to_string(index=False))
print()

if tuned_cv >= cat_cv:
    final_proba = test_tuned
    chosen = 'Tuned CatBoost (OOF=' + str(round(tuned_cv, 5)) + ')'
else:
    final_proba = test_cat
    chosen = 'Baseline CatBoost (OOF=' + str(round(cat_cv, 5)) + ')'

print('Using:', chosen)

In [ ]:
submission = pd.DataFrame({
    'id'             : test['id'].values,
    'Irrigation_Need': [target_inv[i] for i in final_proba.argmax(1)]
})

OUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
submission.to_csv(f'{OUT_DIR}/submission.csv', index=False)

print('Saved:', OUT_DIR + '/submission.csv  (' + str(len(submission)) + ' rows)')
print()
print('Prediction distribution:')
print(submission['Irrigation_Need'].value_counts())
print()
print('Expected (approx from train):')
print(train[TARGET].value_counts(normalize=True).round(4))
submission.head(10)

In [ ]:
assert list(submission.columns) == ['id', 'Irrigation_Need'], 'Column mismatch'
assert len(submission) == len(sub), f'Row count mismatch: {len(submission)} != {len(sub)}'
assert set(submission['Irrigation_Need'].unique()) <= {'Low', 'Medium', 'High'}, 'Unknown class'
print('All sanity checks passed! Ready to submit.')